# CropSmart NAFSI Track 1 — Full Pipeline Orchestration

> **Disclaimer — AI-Generated Quick-Start Notebook**
>
> This notebook is a **one-stop, all-in-one** orchestrator that reproduces our entire analysis
> from raw data acquisition through final results for **Tasks 1–4**. Each section explains
> what is happening under the hood, which tools and datasets are involved, and roughly how
> long each step takes. **Just run the cells in order.**
>
> However, this is intentionally a high-level flyover. For a **more granular understanding**
> of the methodology, assumptions, and interpretation behind each step, we strongly recommend:
>
> 1. **Read the [README](README.md)** for the full project layout, environment setup, and data-tier explanation.
> 2. **Run each task's notebooks individually** — they live under `notebooks/task1_ndvi_timeseries/`,
>    `notebooks/task2_crop_rotation/`, and `notebooks/task3_soil_moisture/`, with detailed markdown
>    commentary between every code section.
> 3. **Read the accompanying paper** for full methodology, results, and discussion.

---

## Challenge Context

This project uses three operational geospatial datasets to analyse vegetation dynamics,
detect crop rotation patterns, quantify soil moisture anomalies, and build predictive
models for crop type classification across the U.S. Corn Belt.

| Dataset | Description | Native Resolution | Period |
|---|---|---|---|
| **CDL** (Cropland Data Layer) | Annual crop-type classification from USDA NASS | 30 m | 2008–2025 |
| **MODIS NDVI** | Weekly composite vegetation index | 250 m | 2000–2026 |
| **SMAP L4** | Weekly surface soil moisture | 9 km | 2015–2025 |

All three are served through the [CropSmart WMS/WPS](https://cloud.csiss.gmu.edu/CropSmart-documentation)
endpoints and are downloaded programmatically by our pipeline.

**Study area:** 13-state Corn Belt (IA, NE, IL, IN, OH, MN, MO, SD, WI, KS, ND, KY, MI),
reprojected to EPSG:5070 (CONUS Albers Equal-Area) at ~557 m analysis resolution.

---

## Step 0 — Environment Setup

Before anything else we need the Python dependencies. The cell below runs
`pip install -r requirements.txt`, which pulls in the full scientific stack
(NumPy, pandas, xarray, rasterio, geopandas, matplotlib, JAX, NumPyro,
LightGBM, and more — see the file for the complete list).

If you are on a fresh environment you may also need to register the local
`src/` package so imports work:

```
pip install -e .
```

> **Time estimate:** ~1–3 minutes depending on your network and whether wheels are cached.

In [ ]:
import subprocess, sys

print("Installing dependencies from requirements.txt ...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print("All dependencies installed successfully.")
else:
    print("pip install failed — check the output below:")
    print(result.stdout[-1000:])
    print(result.stderr[-1000:])

# Also ensure the local src package is importable
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "-q"],
    capture_output=True, text=True,
)
print("src/ package registered (editable install).")

---

## Step 1 — Data Acquisition & Processing

Our data pipeline has three stages, each handled by a dedicated script:

| Stage | Script | What it does | Output |
|-------|--------|-------------|--------|
| **Download** | `scripts/download_data.py` | Fetches raw GeoTIFFs from CropSmart / CropScape WMS endpoints for CDL (2008–2025), NDVI growing-season weeks (2015–2024), and SMAP weekly composites (2015–2025). | `data/raw/{cdl,ndvi,smap}/` |
| **Build Interim** | `scripts/build_interim_data.py` | Stacks per-year GeoTIFFs into analysis-ready NetCDF cubes with dimensions `(year, y, x)` or `(time, y, x)`. | `data/interim/{cdl,ndvi,smap}/` |
| **Export Parquet** | `scripts/process_interim_to_parquet.py` | Converts each NetCDF stack into a wide-format Parquet table (columns = weeks, rows = pixels) with a JSON sidecar carrying the CRS and affine transform. This is what all notebooks consume. | `data/processed/{cdl,ndvi,smap}/` |

The cell below runs all three stages sequentially. Each stage prints progress as it goes.

> **Time estimate:** This is the longest step. Downloading depends heavily on your network
> speed — expect **30–90 minutes** for the full dataset on a typical broadband connection
> (much faster if running on the CropSmart JupyterHub). The interim-build and Parquet-export
> steps are I/O-bound and typically finish in **5–15 minutes** each.
>
> **Note:** The scripts are *idempotent* — they skip files that already exist on disk, so
> re-running is safe and fast if you've already completed a partial download.

In [ ]:
import subprocess, sys, time

data_steps = [
    ("1/5  Downloading raw CDL GeoTIFFs ...",
     [sys.executable, "scripts/download_data.py", "--dataset", "cdl"]),
    ("2/5  Downloading raw NDVI GeoTIFFs (growing-season weeks) ...",
     [sys.executable, "scripts/download_data.py", "--dataset", "ndvi"]),
    ("3/5  Downloading raw SMAP GeoTIFFs (all 52 weeks per year) ...",
     [sys.executable, "scripts/download_data.py", "--dataset", "smap"]),
    ("4/5  Building interim NetCDF stacks from raw GeoTIFFs ...",
     [sys.executable, "scripts/build_interim_data.py", "--dataset", "all"]),
    ("5/5  Exporting processed Parquet tables ...",
     [sys.executable, "scripts/process_interim_to_parquet.py", "--dataset", "all"]),
]

for label, cmd in data_steps:
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    if proc.returncode == 0:
        print(f"  Finished in {elapsed:.0f}s")
        if proc.stdout.strip():
            # show last few lines of stdout as a summary
            for line in proc.stdout.strip().splitlines()[-5:]:
                print(f"    {line}")
    else:
        print(f"  FAILED after {elapsed:.0f}s")
        print(proc.stderr[-800:] if proc.stderr else proc.stdout[-800:])

print("\nData pipeline complete.  Processed Parquet files are under data/processed/.")

---

## Step 2 — Task 1: NDVI Time-Series Phenology Analysis

Task 1 compares the seasonal vegetation curves (phenology) of **corn vs. soybean** across
the Corn Belt using MODIS NDVI and CDL crop masks.

The analysis runs across three notebooks:

| Notebook | Purpose |
|----------|---------|
| `01_data_ingestion_cdl_ndvi.ipynb` | Loads CDL Parquet, applies purity filtering, and joins NDVI weekly composites to build a per-pixel panel of NDVI time series with crop labels. |
| `02_ndvi_phenology_multi_year_cdl.ipynb` | Computes empirical phenological features (green-up date, peak NDVI, senescence) for corn and soybean across multiple years and visualises their distributions. |
| `03_ndvi_phenology_hsgp_bayesian.ipynb` | Fits a **Hilbert-Space Gaussian Process (HSGP)** model via NumPyro to produce smooth, uncertainty-aware phenology curves for each crop, with posterior-predictive credible intervals. |

The third notebook involves Bayesian MCMC sampling on JAX, which is the most
compute-intensive step in the entire project.

> **Time estimate:** Notebooks 01 and 02 run in **2–5 minutes**. Notebook 03 (HSGP)
> can take **15–45 minutes** depending on your hardware (GPU-accelerated JAX is much faster).

In [ ]:
import subprocess, sys, time, glob
from pathlib import Path
from IPython.display import display, Image, Markdown

TASK1_DIR = Path("notebooks/task1_ndvi_timeseries")
TASK1_NOTEBOOKS = [
    "01_data_ingestion_cdl_ndvi.ipynb",
    "02_ndvi_phenology_multi_year_cdl.ipynb",
    "03_ndvi_phenology_hsgp_bayesian.ipynb",
]

print("Running Task 1 notebooks (NDVI Phenology)\n")
for nb_name in TASK1_NOTEBOOKS:
    nb_path = TASK1_DIR / nb_name
    print(f"  Running {nb_name} ...")
    t0 = time.time()
    proc = subprocess.run(
        [sys.executable, "-m", "jupyter", "nbconvert",
         "--to", "notebook", "--execute", "--inplace",
         "--ExecutePreprocessor.timeout=3600",
         str(nb_path)],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    status = "done" if proc.returncode == 0 else "FAILED"
    print(f"    {status} ({elapsed:.0f}s)")
    if proc.returncode != 0:
        print(f"    {proc.stderr[-400:]}")

# Display key result figures
print("\n--- Task 1 Results ---\n")
fig_dir = Path("artifacts/figures/task1")
key_figs = ["hsgp_phenology_crops.png", "hsgp_phenology_corn_vs_soy.png",
            "phenological_features_by_year.png", "calibration_diagnostics.png"]

for fname in key_figs:
    fpath = fig_dir / fname
    if fpath.exists():
        display(Markdown(f"**{fname}**"))
        display(Image(filename=str(fpath), width=800))
    else:
        print(f"  (figure not found: {fname})")

---

## Step 3 — Task 2: Crop Rotation Pattern Identification

Task 2 analyses 10 years of CDL data (2015–2024) to classify every corn/soy pixel
in the Corn Belt into one of three rotation patterns: **regular rotation**, **monoculture**,
or **irregular**.

The analysis runs across four notebooks:

| Notebook | Purpose |
|----------|---------|
| `01_cdl_timeseries_loading.ipynb` | Loads the CDL wide Parquet, filters to corn/soy eligible pixels (appearing in at least 5 of 10 years), and recodes crop sequences into a `{corn, soy, other}` alphabet. |
| `02_rotation_metrics_computation.ipynb` | Computes four sequence metrics per pixel — **alternation score**, **pattern edit distance**, **Shannon entropy**, and **maximum run length** — plus Markov transition probabilities and an optional Bayesian **Dirichlet-Multinomial** layer. |
| `03_rotation_classification.ipynb` | Applies a hierarchical rule-based classifier using YAML-driven thresholds, runs a sensitivity sweep, and generates Bayesian DM rasters (posterior P(regular), P(mono), uncertainty). |
| `04_spatial_maps_and_areal_export.ipynb` | Produces publication-ready maps (raw + 3×3 majority-filtered rotation classes, DM probability surfaces), county-level summaries, and per-state areal statistics CSVs. |

All thresholds and parameters are defined in `configs/task2_crop_rotation.yaml`.

> **Time estimate:** Task 2 is mostly vectorised pandas/NumPy operations on a ~1.3 M pixel panel.
> The full four-notebook sequence typically completes in **5–15 minutes**.

In [ ]:
import subprocess, sys, time
from pathlib import Path
from IPython.display import display, Image, Markdown
import pandas as pd

TASK2_DIR = Path("notebooks/task2_crop_rotation")
TASK2_NOTEBOOKS = [
    "01_cdl_timeseries_loading.ipynb",
    "02_rotation_metrics_computation.ipynb",
    "03_rotation_classification.ipynb",
    "04_spatial_maps_and_areal_export.ipynb",
]

print("Running Task 2 notebooks (Crop Rotation)\n")
for nb_name in TASK2_NOTEBOOKS:
    nb_path = TASK2_DIR / nb_name
    print(f"  Running {nb_name} ...")
    t0 = time.time()
    proc = subprocess.run(
        [sys.executable, "-m", "jupyter", "nbconvert",
         "--to", "notebook", "--execute", "--inplace",
         "--ExecutePreprocessor.timeout=3600",
         str(nb_path)],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    status = "done" if proc.returncode == 0 else "FAILED"
    print(f"    {status} ({elapsed:.0f}s)")
    if proc.returncode != 0:
        print(f"    {proc.stderr[-400:]}")

# Display key result figures
print("\n--- Task 2 Results ---\n")
fig_dir = Path("artifacts/figures/task2")
key_figs = [
    "task2__rotation_map__smoothed__20260412.png",
    "task2__metric_histograms.png",
    "task2__rotation_dm_p_regular__20260412.png",
    "task2__per_state_rotation_classes.png",
]

for fname in key_figs:
    fpath = fig_dir / fname
    if fpath.exists():
        display(Markdown(f"**{fname}**"))
        display(Image(filename=str(fpath), width=800))
    else:
        print(f"  (figure not found: {fname})")

# Show summary table if available
tbl_dir = Path("artifacts/tables/task2")
for csv in sorted(tbl_dir.glob("*.csv")):
    display(Markdown(f"**{csv.name}**"))
    display(pd.read_csv(csv).head(20))

---

## Step 4 — Task 3: Soil Moisture Anomaly Detection

Task 3 uses SMAP L4 weekly soil moisture to detect anomalous conditions during two
real-world agricultural stress events:

| Event | Period | What happened |
|-------|--------|---------------|
| **2019 Midwest Flood** | Apr–Jul 2019 | Prolonged excess soil moisture across the Mississippi / Missouri River valleys delayed planting and damaged crops. |
| **2022 Plains Drought** | Jun–Sep 2022 | Flash drought across Kansas, Nebraska, and the Southern Plains — extreme heat combined with low precipitation. |

The analysis runs across three notebooks:

| Notebook | Purpose |
|----------|---------|
| `01_pixel_panel_smap_cdl.ipynb` | Defines the spatial subset (Task 2's eligible corn/soy pixels), attaches event-year CDL crop labels, and joins with SMAP weekly Parquet. |
| `02_climatology_and_anomalies.ipynb` | Computes a per-pixel, per-week baseline climatology (2015–2021), then calculates both **frequentist z-score** anomalies and **Bayesian Normal-Inverse-Gamma (NIG)** posterior-predictive anomaly probabilities for each event window. |
| `03_maps_timeseries_tables.ipynb` | Generates six figures per event (4-panel anomaly maps, time series, NIG P(drought) maps, scatter diagnostics, uncertainty maps, duration fraction maps) plus state-by-crop summary tables. |

The Bayesian NIG framework provides a principled way to handle short baselines — it yields
a full Student-t predictive distribution, so we get calibrated P(drought) and P(wet) scores
rather than just point z-scores.

> **Time estimate:** Task 3 processes ~1.3 M pixels × ~20 event weeks. The full sequence
> typically completes in **5–10 minutes**.

In [ ]:
import subprocess, sys, time
from pathlib import Path
from IPython.display import display, Image, Markdown
import pandas as pd

TASK3_DIR = Path("notebooks/task3_soil_moisture")
TASK3_NOTEBOOKS = [
    "01_pixel_panel_smap_cdl.ipynb",
    "02_climatology_and_anomalies.ipynb",
    "03_maps_timeseries_tables.ipynb",
]

print("Running Task 3 notebooks (Soil Moisture Anomalies)\n")
for nb_name in TASK3_NOTEBOOKS:
    nb_path = TASK3_DIR / nb_name
    print(f"  Running {nb_name} ...")
    t0 = time.time()
    proc = subprocess.run(
        [sys.executable, "-m", "jupyter", "nbconvert",
         "--to", "notebook", "--execute", "--inplace",
         "--ExecutePreprocessor.timeout=3600",
         str(nb_path)],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    status = "done" if proc.returncode == 0 else "FAILED"
    print(f"    {status} ({elapsed:.0f}s)")
    if proc.returncode != 0:
        print(f"    {proc.stderr[-400:]}")

# Display key result figures
print("\n--- Task 3 Results ---\n")
fig_dir = Path("artifacts/figures/task3")
key_figs = [
    "task3__midwest_flood_2019__nig_p_drought_4panel__20260412.png",
    "task3__plains_drought_2022__nig_p_drought_4panel__20260412.png",
    "task3__plains_drought_2022__zscore_vs_nig_scatter__20260412.png",
    "task3__midwest_flood_2019__anomaly_timeseries_cropland__20260412.png",
]

for fname in key_figs:
    fpath = fig_dir / fname
    if fpath.exists():
        display(Markdown(f"**{fname}**"))
        display(Image(filename=str(fpath), width=800))
    else:
        print(f"  (figure not found: {fname})")

# Show summary tables
tbl_dir = Path("artifacts/tables/task3")
for csv in sorted(tbl_dir.glob("*anomaly_stats*.csv")):
    display(Markdown(f"**{csv.name}**"))
    display(pd.read_csv(csv).head(20))

print("\nTask 3 complete. Continuing to Task 4 below.")

---

## Step 5 — Task 4: Crop Type Classification (LightGBM)

Task 4 brings everything together: given a pixel's **CDL planting history** (Task 2 features),
**NDVI phenology** (Task 1 features), and **SMAP soil moisture** (Task 3 features), predict
which crop was planted in a held-out future year.

The analysis runs across four notebooks:

| Notebook | Purpose |
|----------|---------|
| `01_feature_panel_construction.ipynb` | Builds a 5M-row feature matrix (38 features per pixel-year) from CDL lag codes, rotation metrics, NDVI phenological descriptors, and SMAP growing-season statistics. Exports train/val panel and a separate 2023 test frame. |
| `02_model_training_and_ablation.ipynb` | Trains a LightGBM classifier under four ablation configurations (CDL-only → +NDVI → +SMAP → full) to quantify each data source's value. Evaluates on the 2023 test year. |
| `03_feature_importance_and_regime_analysis.ipynb` | Runs SHAP TreeExplainer on the trained model and stratifies performance by rotation regime (regular / monoculture / irregular). |
| `04_spatial_maps_and_discussion.ipynb` | Generates predicted vs. true crop-type maps with state boundary overlays and an agreement/disagreement map. |

> **Time estimate:** Notebook 01 (feature engineering) takes **5–10 minutes**; Notebook 02
> (model training with 4 ablations + test eval) takes **5–15 minutes**; Notebooks 03 and 04
> run in **2–5 minutes** each.
>
> **Optional — Hyperparameter Tuning:** An additional notebook `02b_hyperparameter_tuning.ipynb`
> uses Optuna to search for better LightGBM hyperparameters. This is **not run by default**
> here because it takes ~30 minutes. You can run it manually afterwards for potential
> accuracy improvements.

In [ ]:
import subprocess, sys, time
from pathlib import Path
from IPython.display import display, Image, Markdown
import pandas as pd

TASK4_DIR = Path("notebooks/task4_crop_mapping")
TASK4_NOTEBOOKS = [
    "01_feature_panel_construction.ipynb",
    "02_model_training_and_ablation.ipynb",
    "03_feature_importance_and_regime_analysis.ipynb",
    "04_spatial_maps_and_discussion.ipynb",
]

print("Running Task 4 notebooks (Crop Type Classification)\n")
for nb_name in TASK4_NOTEBOOKS:
    nb_path = TASK4_DIR / nb_name
    print(f"  Running {nb_name} ...")
    t0 = time.time()
    proc = subprocess.run(
        [sys.executable, "-m", "jupyter", "nbconvert",
         "--to", "notebook", "--execute", "--inplace",
         "--ExecutePreprocessor.timeout=3600",
         str(nb_path)],
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0
    status = "done" if proc.returncode == 0 else "FAILED"
    print(f"    {status} ({elapsed:.0f}s)")
    if proc.returncode != 0:
        print(f"    {proc.stderr[-400:]}")

# Display key result figures
print("\n--- Task 4 Results ---\n")
fig_dir = Path("artifacts/figures/task4")
key_figs = [
    "task4_ablation_comparison.png",
    "task4_test_confusion_matrix.png",
    "task4_shap_importance.png",
    "task4_crop_maps_pred_vs_true.png",
    "task4_agreement_map.png",
]

for fname in key_figs:
    fpath = fig_dir / fname
    if fpath.exists():
        display(Markdown(f"**{fname}**"))
        display(Image(filename=str(fpath), width=800))
    else:
        print(f"  (figure not found: {fname})")

# Show key tables
tbl_dir = Path("artifacts/tables/task4")
for tbl_name in ["task4_ablation_results.csv", "task4_regime_stratified_metrics.csv"]:
    tbl_path = tbl_dir / tbl_name
    if tbl_path.exists():
        display(Markdown(f"**{tbl_name}**"))
        display(pd.read_csv(tbl_path))

# Show test metrics JSON
import json
for jpath in sorted(tbl_dir.glob("task4__test_metrics__*.json")):
    metrics = json.loads(jpath.read_text(encoding="utf-8"))
    display(Markdown(f"**Test-Year Metrics ({jpath.name})**"))
    print(f"  Overall Accuracy: {metrics['overall_accuracy']:.4f}")
    print(f"  Macro F1:         {metrics['macro_f1']:.4f}")
    for cn, f1 in metrics.get("per_class_f1_named", {}).items():
        print(f"  F1 {cn}: {f1:.4f}")

print("\nDone. All Task 1-4 results have been generated.")
print("See artifacts/figures/ and artifacts/tables/ for the full set of outputs.")